In [ ]:
#Creating DT Data Pt. 1
import pandas as pd

gold_path  = "../data/Q1_GOLD_LEFT_V_PLACEMENT.csv"
units_path = "../data/Q3_UNITS_V_PLACEMENT.csv"

gold_df  = pd.read_csv(gold_path)
units_df = pd.read_csv(units_path)

gold_df["gold_left"] = pd.to_numeric(gold_df["gold_left"], errors="coerce")
gold_df["placement"] = pd.to_numeric(gold_df["placement"], errors="coerce")

units_df["tier"]   = pd.to_numeric(units_df["tier"],   errors="coerce")
units_df["rarity"] = pd.to_numeric(units_df["rarity"], errors="coerce")

gold_df  = gold_df.drop_duplicates(subset=["match_id", "puuid", "gold_left", "placement"])

units_agg = (
    units_df
    .groupby(["match_id", "puuid"], as_index=False)
    .agg(
        sum_unit_tier      = ("tier",   "sum"),   
        mean_unit_rarity   = ("rarity", "mean")   
    )
)

result_df = (
    gold_df[["match_id", "puuid", "gold_left", "placement"]]
    .merge(units_agg, on=["match_id", "puuid"], how="inner")
)

result_df = result_df[
    ["match_id", "puuid", "gold_left",
     "sum_unit_tier", "mean_unit_rarity", "placement"]
]

result_df.to_csv("../data/DT_data_1.csv", index=False)

print(result_df.head())


         match_id                                              puuid  \
0  NA1_5214474372  VKwva7riXrRpNn_pQCF7Gx63CYKG4PxcbND2JnL3Tcceb6...   
1  NA1_5214474372  CSZABWFZly3iWPGGobYZhYk9knB5J0-pUQbO02Zx85ShQv...   
2  NA1_5214474372  B_mWPVRiuzGR-eUTN6CDIwkjbf_PXOmitrSgK3r-Yo38gK...   
3  NA1_5214474372  _W30FKv18DHc61O6J_7rVUV5nrp779aWIE3MSuA0YS9Mcu...   
4  NA1_5214474372  -rQ5Ep_GYZG2QjV9oB3GnGS494CbFg7oZWjGYLBKMW9efl...   

   gold_left  sum_unit_tier  mean_unit_rarity  placement  
0         40             14          2.375000          5  
1          0             10          2.250000          8  
2          4             13          4.333333          6  
3          0             18          3.090909          4  
4          1             15          2.750000          7  


In [7]:
# Creating Tree Data Pt. 2


units_path = "../data/Q3_UNITS_V_PLACEMENT.csv"
units_df   = pd.read_csv(units_path)


units_unique = units_df.drop_duplicates(
    subset=["match_id", "puuid", "unit_character_id"]
).copy()
units_unique["fielded"] = 1      

unit_matrix = (
    units_unique
    .pivot_table(
        index=["match_id", "puuid"],
        columns="unit_character_id",
        values="fielded",
        aggfunc="max",          
        fill_value=0            
    )
)

unit_matrix.columns = unit_matrix.columns.get_level_values(0)


placement_df = (
    units_df[["match_id", "puuid", "placement"]]
      .drop_duplicates(subset=["match_id", "puuid"])
      .set_index(["match_id", "puuid"])
)


final_df = pd.concat([unit_matrix, placement_df], axis=1).reset_index()

cols = [c for c in final_df.columns if c not in ("placement", "match_id", "puuid")]
final_df = final_df[["match_id", "puuid"] + sorted(cols) + ["placement"]]


final_df.to_csv("../data/DT_data_2.csv", index=False)
print(final_df.head())

         match_id                                              puuid  \
0  NA1_5207953898  1x8SiuU29wBehu4HwldU0tiLaoeYZKkyJatYg12fu8a56P...   
1  NA1_5207953898  23nlj-mOFFeTPgkfSkE7oM0vXUkDvtCrUT9qt-NwnnhWav...   
2  NA1_5207953898  43332cvV4A2mqb2W0JfRmsfY9djf-VGUgRyPY4wB0BLbtw...   
3  NA1_5207953898  7iIW7cDfdPzm9o3z0uea93AooftwLNn9OHAvQ79gKJOUHG...   
4  NA1_5207953898  FCVNYmfyGJb7c4xAU_CDzVjrFzXapffhOPPn6lXfE8Cmtt...   

   TFT13_Akali  TFT13_Ambessa  TFT13_Amumu  TFT13_Beardy  TFT13_Blitzcrank  \
0            0              0            0             1                 0   
1            0              1            0             0                 0   
2            0              0            0             0                 0   
3            0              1            0             0                 0   
4            0              0            0             0                 0   

   TFT13_Blue  TFT13_Caitlyn  TFT13_Camille  ...  TFT13_Vladimir  \
0           0              1  

In [8]:
#Cleaning Data

df1 = pd.read_csv("../data/DT_data_1.csv")
df2 = pd.read_csv("../data/DT_data_2.csv")

cols_to_drop = ["match_id", "puuid"]

df1 = df1.drop(columns=cols_to_drop, errors="ignore")
df2 = df2.drop(columns=cols_to_drop, errors="ignore")


df1.to_csv("../data/DT_data_1_cleaned.csv", index=False)
df2.to_csv("../data/DT_data_2_cleaned.csv", index=False)

# quick sanity-check
print(df1.head())
print(df2.head())

   gold_left  sum_unit_tier  mean_unit_rarity  placement
0         40             14          2.375000          5
1          0             10          2.250000          8
2          4             13          4.333333          6
3          0             18          3.090909          4
4          1             15          2.750000          7
   TFT13_Akali  TFT13_Ambessa  TFT13_Amumu  TFT13_Beardy  TFT13_Blitzcrank  \
0            0              0            0             1                 0   
1            0              1            0             0                 0   
2            0              0            0             0                 0   
3            0              1            0             0                 0   
4            0              0            0             0                 0   

   TFT13_Blue  TFT13_Caitlyn  TFT13_Camille  TFT13_Cassiopeia  TFT13_Chainsaw  \
0           0              1              1                 0               0   
1           0              

In [10]:
#Training and Testing Data - Pt 1
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/DT_data_1_cleaned.csv")

X = df.drop(columns=["placement"])
y = df["placement"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y 
)

X_train.to_csv("../data/DT_1_training_data.csv", index=False)
X_test.to_csv("../data/DT_1_testing_data.csv",  index=False)

y_train.to_csv("../data/DT_1_training_labels.csv", index=False, header=["placement"])
y_test.to_csv("../data/DT_1_testing_labels.csv",  index=False, header=["placement"])

In [ ]:
#Training and Testing Data - Pt 1
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/DT_data_1_cleaned.csv")

X = df.drop(columns=["placement"])
y = df["placement"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y 
)

X_train.to_csv("../data/DT_1_training_data.csv", index=False)
X_test.to_csv("../data/DT_1_testing_data.csv",  index=False)

y_train.to_csv("../data/DT_1_training_labels.csv", index=False, header=["placement"])
y_test.to_csv("../data/DT_1_testing_labels.csv",  index=False, header=["placement"])

In [ ]:
#Training and Testing Data - Pt 2

df = pd.read_csv("../data/trait_counts_cleaned.csv")

X = df.drop(columns=["top_four"])
y = df["top_four"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y 
)

X_train.to_csv("../data/DT_2_training_data.csv", index=False)
X_test.to_csv("../data/DT_2_testing_data.csv",  index=False)

y_train.to_csv("../data/DT_2_training_labels.csv", index=False, header=["top_four"])
y_test.to_csv("../data/DT_2_testing_labels.csv",  index=False, header=["top_four"])

In [12]:
#Training and Testing Data - Pt 3
df = pd.read_csv("../data/DT_data_2_cleaned.csv")

X = df.drop(columns=["placement"])
y = df["placement"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y 
)

X_train.to_csv("../data/DT_3_training_data.csv", index=False)
X_test.to_csv("../data/DT_3_testing_data.csv",  index=False)

y_train.to_csv("../data/DT_3_training_labels.csv", index=False, header=["placement"])
y_test.to_csv("../data/DT_3_testing_labels.csv",  index=False, header=["placement"])